# Graphs at every strict dismantling level

This notebook accompanies **Linear-size graphs at every strict dismantling level**. It follows the constructions, explains the finite evidence, and runs the same calculations as the local reproduction guide.

The family $F_k$ has $3k+8$ vertices, the family $H_k$ has $4k+4$, and the supplementary family $J_k$ has $6k+6$. Each graph has strict dismantling level $k$, for every $k\geq1$. The proofs are in the [mathematical supplement](https://github.com/Geometric-and-Combinatorial-Structures/strictly-k-dismantlable-graphs/blob/v1.0.0/supplement/supplement.pdf); the [article reference](https://github.com/Geometric-and-Combinatorial-Structures/strictly-k-dismantlable-graphs/blob/main/ARTICLE.md) is maintained separately.

Choose **Runtime → Run all** in Colab. The notebook uses release **v1.0.0**. Each calculation creates a new output directory and preserves the inputs and earlier results. Python 3.11 or later and its standard library suffice for all calculations.


In [ ]:
#@title Obtain release v1.0.0
from pathlib import Path
import json, os, subprocess, sys, tempfile, urllib.request, zipfile

RELEASE = "v1.0.0"
RELEASE_URL = "https://github.com/Geometric-and-Combinatorial-Structures/strictly-k-dismantlable-graphs/releases/download/v1.0.0/strictly-k-dismantlable-graphs-v1.0.0.zip"
work = Path(tempfile.mkdtemp(prefix="strict-dismantling-"))
archive = work / "release.zip"
# The optional override lets the release tests use their exact local archive.
url = os.environ.get("STRICT_DISMANTLING_ARCHIVE", RELEASE_URL)
with urllib.request.urlopen(url) as response, archive.open("wb") as output:
    output.write(response.read())
with zipfile.ZipFile(archive) as zipped:
    for member in zipped.namelist():
        destination = (work / member).resolve()
        if work not in destination.parents:
            raise ValueError("Archive member outside the extraction directory")
    zipped.extractall(work)
repo = work / "strictly-k-dismantlable-graphs-v1.0.0"
if not (repo / "reproduce.py").is_file():
    raise ValueError("The archive does not contain the expected release")
print("Release", RELEASE, "is ready.")


## Verify the finite hypotheses

A deletion at level $k$ requires both a neighbourhood in $\mathcal D_{k-1}$ and a remainder that can still be reduced at level $k$. The positive witnesses justify these two requirements. A negative certificate excludes every possible first deletion.

The next cell verifies all required finite data, including the folds, recursive certificates, clique tables, figure incidences and homology calculations. It also checks independent constructions and deliberate alterations. The first check compares every distributed file with its checksum.


In [ ]:
result_directory = work / "required-calculations"
subprocess.run([sys.executable, str(repo / "reproduce.py"), "--output", str(result_directory)], check=True)
report = json.loads((result_directory / "report.json").read_text(encoding="utf-8"))
if report["status"] != "PASS":
    raise RuntimeError("The required calculations did not complete successfully")
results = {item["id"]: item["details"] for item in report["checks"]}


## The eleven-vertex seed of $F_k$

The seed $F_1$ has eleven vertices and thirty-three edges. Its reduction at level one deletes ten vertices; the twenty-three inner folds justify the required neighbourhood tests. The two forced folds at level zero end in a graph with nine vertices and no available fold, excluding level zero.

The first extension $F_2$ has fourteen vertices and sixty-five edges. Deleting its centre leaves a graph reduced by twelve folds. The thirteen other neighbourhoods have non-zero alternating clique sums, which exclude first deletions at the lower level. Appendix A of the supplement supplies the complete proof.


In [ ]:
a = results["article"]
print("Seed:", a["seed"]["vertices"], "vertices;", a["seed"]["edges"], "edges")
print("Outer deletions:", a["D1_reduction"]["outer_deletions"])
print("Inner folds:", a["D1_reduction"]["inner_folds"])
print("Remainder folds:", a["remainder"]["fold_count"])
print("Non-central clique rows:", a["clique_counts"]["rows_checked"])


## The $4k+4$ and $6k+6$ families

For $H_k$, the first two members are induced subgraphs of the sixteen-vertex seed. Seven certificate roots prove their strict levels, the strict level of the seed, and the required level of the first remainder. Nineteen non-zero clique sums then provide the remaining hypotheses of the construction theorem.

For $J_k$, both positive reductions are displayed explicitly in the supplement. Every inner fold is checked in its current neighbourhood. The degree table excludes a first fold in the seed, and seventeen clique rows exclude the non-central first deletions in its extension.


In [ ]:
print("H_k:", results["4k4"]["roots"], "certificate roots and", results["4k4"]["clique_rows"], "clique rows")
print("J_k:", results["6k6"])


## Kernel completion and reduced homology

Remove the edge $\{1,2\}$ from $F_1$ to obtain its kernel. The kernel is acyclic over $\mathbb F_2$, while its neighbourhoods have non-zero reduced homology in degree one. Completing the edge gives the seed, and the neighbourhood homology in the first extension is shifted by one degree.

The empty graph is retained in the augmented convention: its clique complex consists only of the empty face, with $\widetilde\beta_{-1}=1$. The example $C_4\sqcup K_1$ shows why a zero Euler obstruction does not imply acyclicity: its non-zero groups in degrees zero and one have cancelling contributions.


In [ ]:
print("Kernel edges:", results["kernels"]["kernel_edges"])
print("Kernel reduced Betti numbers:", results["kernels"]["kernel_reduced_betti"])
for row in results["kernels"]["links"]:
    print("Vertex", row["vertex"], ":", row["kernel_link_betti"], "becomes", row["extension_link_betti"])
print("Boundary examples:", results["homology"])


## Additional finite checks

The extended calculation considers every labelled seed on one, two or three vertices, all profiles for one or two auxiliary types, and all type graphs, at zero to three layers. Homology is checked for seeds of order at most two. It also checks all covered profile pairs for the kernel identities on seeds of order at most three.

The output gives the exact finite ranges and counts. These calculations test the identities on those sets. The proofs in the supplement establish them without a bound on the number of layers.


In [ ]:
extended_directory = work / "extended-calculations"
subprocess.run([sys.executable, str(repo / "reproduce.py"), "--extended", "--output", str(extended_directory)], check=True)
extended = json.loads((extended_directory / "report.json").read_text(encoding="utf-8"))
for item in extended["checks"]:
    if item["id"] in ("templates", "kernel-identities"):
        print(item["title"])
        print(item["details"])


## Construct an example

Finally, construct $F_5$. It has twenty-three vertices and 215 edges. The generated description matches the numerical edge labels to the seed vertices, centres and auxiliaries, so the example can be used directly in further mathematical work.


In [ ]:
example_directory = work / "F5"
subprocess.run([sys.executable, str(repo / "generate.py"), "--family", "3k8", "--k", "5", "--output", str(example_directory)], check=True)
print((example_directory / "description.txt").read_text(encoding="utf-8"))
print("All reports and the example are in:", work)


The files remain available for the duration of the notebook session. In Colab, open the **Files** panel and navigate to the printed directory to download the reports or the example. The released inputs are unchanged. For the proof behind any calculation, return to the [supplement](https://github.com/Geometric-and-Combinatorial-Structures/strictly-k-dismantlable-graphs/blob/v1.0.0/supplement/supplement.pdf) or the [result-by-result guide](https://github.com/Geometric-and-Combinatorial-Structures/strictly-k-dismantlable-graphs/blob/v1.0.0/docs/results.md).
